# Text EDA — Davidson Hate Speech & Offensive Language Dataset

**Purpose:** load the raw text dataset, inspect its real structure, clean it, and run the
EDA required by `master_plan.md` section 5 (class distribution, text length distribution,
top words/n-grams per class, word clouds) before any modeling starts.

### Getting the dataset (all free, pick one)
1. **Kaggle:** https://www.kaggle.com/mrmorj/hate-speech-and-offensive-language-dataset — download `labeled_data.csv`, place it at `data/text/labeled_data.csv`.
2. **HuggingFace Datasets:** `from datasets import load_dataset; ds = load_dataset('tdavidson/hate_speech_offensive')` then export to CSV.
3. **Direct from the original author's GitHub repo (raw CSV, no account needed):**
   `https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv`

Whichever you use, save the file as `data/text/labeled_data.csv` (this path is gitignored —
see `.gitignore` — since we don't redistribute the raw dataset in the repo itself, per the
plan's reproducibility rule: instructions to re-download, not a copy of the data).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so `from src.preprocessing import ...` works from notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import clean_text, ensure_nltk_resources
ensure_nltk_resources()  # one-time download of punkt/wordnet/pos-tagger data

sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', 120)

## 1. Load and inspect raw structure
We verify the actual columns/shape before assuming anything — same discipline the plan
calls for on the multimodal dataset, applied here too.

In [ ]:
DATA_PATH = '../data/text/labeled_data.csv'
df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
print('columns:', list(df.columns))
df.head()

In [ ]:
# Davidson et al. schema: 'class' is the label (0=hate speech, 1=offensive language,
# 2=neither), 'tweet' is the raw text, plus annotator vote-count columns we don't need
# for modeling. We confirm this matches what we actually loaded before renaming anything.
df.info()
df.isna().sum()

In [ ]:
LABEL_MAP = {0: 'Hate Speech', 1: 'Offensive Language', 2: 'Neither'}
df = df.rename(columns={'class': 'label', 'tweet': 'text'})[['text', 'label']].copy()
df['label_name'] = df['label'].map(LABEL_MAP)
df.head()

## 2. Class distribution
This is the imbalance the plan flags: expect 'Offensive Language' to dominate.

In [ ]:
counts = df['label_name'].value_counts()
pct = (counts / len(df) * 100).round(1)
print(counts)
print(pct)

fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_title('Class distribution (raw counts)')
ax.set_ylabel('count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v}\n({pct.values[i]}%)', ha='center')
plt.tight_layout()
plt.savefig('../notebooks/figures_class_distribution.png', dpi=150)
plt.show()

## 3. Text length distribution (per class)

In [ ]:
df['char_len'] = df['text'].astype(str).str.len()
df['word_len'] = df['text'].astype(str).str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.boxplot(data=df, x='label_name', y='char_len', ax=axes[0])
axes[0].set_title('Character length by class')
sns.boxplot(data=df, x='label_name', y='word_len', ax=axes[1])
axes[1].set_title('Word count by class')
plt.tight_layout()
plt.savefig('../notebooks/figures_length_distribution.png', dpi=150)
plt.show()

df.groupby('label_name')[['char_len','word_len']].describe()

## 4. Clean the text
Applies `src/preprocessing.clean_text` (lowercase, strip URLs/mentions, unwrap hashtags,
demojize emojis, POS-aware lemmatize — see the docstring in that file for the reasoning
behind each decision). We keep the raw `text` column alongside `clean_text` so we can always
trace back to the original.

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text', 'label_name']].sample(10, random_state=42)

## 5. Most frequent words / bigrams per class
Using raw counts (not TF-IDF) here since the goal is descriptive EDA, not a model.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, n=20, ngram_range=(1,1)):
    vec = CountVectorizer(ngram_range=ngram_range, min_df=2)
    X = vec.fit_transform(texts)
    freqs = np.asarray(X.sum(axis=0)).ravel()
    vocab = vec.get_feature_names_out()
    order = freqs.argsort()[::-1][:n]
    return pd.DataFrame({'term': vocab[order], 'count': freqs[order]})

for label_name in LABEL_MAP.values():
    subset = df.loc[df['label_name'] == label_name, 'clean_text']
    print(f'--- Top unigrams: {label_name} ---')
    display(top_ngrams(subset, n=15, ngram_range=(1,1)))

In [ ]:
for label_name in LABEL_MAP.values():
    subset = df.loc[df['label_name'] == label_name, 'clean_text']
    print(f'--- Top bigrams: {label_name} ---')
    display(top_ngrams(subset, n=15, ngram_range=(2,2)))

## 6. Word clouds per class
Purely visual/descriptive — requires the `wordcloud` package (in requirements.txt).

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(18,6))
for ax, label_name in zip(axes, LABEL_MAP.values()):
    text_blob = ' '.join(df.loc[df['label_name'] == label_name, 'clean_text'])
    wc = WordCloud(width=500, height=400, background_color='white').generate(text_blob)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(label_name)
    ax.axis('off')
plt.tight_layout()
plt.savefig('../notebooks/figures_wordclouds.png', dpi=150)
plt.show()

## 7. Open decision to make together: class imbalance handling
Per master_plan.md section 5.3, we must pick ONE defensible approach once we've seen the
real class distribution above (class-weighted loss vs. SMOTE/oversampling of the minority
'Hate Speech' class) — **not decided in this notebook**, revisit in the baseline-model step
once these numbers are in front of us.

## 8. Save cleaned dataset
Train/val/test stratified splitting happens in the next phase (baseline model), not here —
EDA runs on the full cleaned set first.

In [ ]:
df.to_csv('../data/text/cleaned_labeled_data.csv', index=False)
print('saved:', df.shape)